# Exploration Dataset — CHEESE-HIDB
Analyse du dataset CR-IDB pour le module Computer Vision Savencia.

In [ ]:
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np

DATASET_ROOT = Path('../ml/sample_images/CHEESE-HIDB-main')
TYPES = ['Extra-Hard', 'Hard', 'Semi-Hard']
CLASSES = ['Target', 'NotTarget']

## Bloc 1 — Inventaire

In [ ]:
rows = []
for t in TYPES:
    for c in CLASSES:
        folder = DATASET_ROOT / t / c
        images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.JPG'))
        rows.append({'type': t, 'classe': c, 'count': len(images)})

df = pd.DataFrame(rows)
pivot = df.pivot(index='type', columns='classe', values='count')
pivot['Total'] = pivot.sum(axis=1)
pivot['Ratio Target%'] = (pivot['Target'] / pivot['Total'] * 100).round(1)
print(pivot)
print(f'\nTotal global : {pivot["Total"].sum()} images')

## Bloc 2 — Qualité visuelle

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Échantillons par type × classe (2 images chacun)', fontsize=14)

for i, t in enumerate(TYPES):
    for j, c in enumerate(CLASSES):
        folder = DATASET_ROOT / t / c
        images = sorted(list(folder.glob('*.jpg')) + list(folder.glob('*.JPG')) + list(folder.glob('*.png')))
        for k in range(2):
            ax = axes[i][j * 2 + k]
            if k < len(images):
                img = mpimg.imread(str(images[k]))
                ax.imshow(img)
                ax.set_title(f'{t}\n{c}', fontsize=8)
            ax.axis('off')

plt.tight_layout()
plt.show()

## Bloc 3 — Propriétés techniques

In [ ]:
stats = []
corrupted = []

for t in TYPES:
    for c in CLASSES:
        folder = DATASET_ROOT / t / c
        images = list(folder.glob('*.jpg')) + list(folder.glob('*.JPG')) + list(folder.glob('*.png'))
        for img_path in images:
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                    size_kb = img_path.stat().st_size / 1024
                    stats.append({'type': t, 'classe': c, 'width': w, 'height': h, 'size_kb': round(size_kb, 1)})
            except Exception as e:
                corrupted.append({'path': str(img_path), 'error': str(e)})

df_stats = pd.DataFrame(stats)
print('=== Dimensions ===')
print(df_stats[['width', 'height', 'size_kb']].describe().round(1))
print(f'\nFichiers corrompus : {len(corrupted)}')
if corrupted:
    print(pd.DataFrame(corrupted))

## Bloc 4 — Distribution colorimétrique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_rgb = ['red', 'green', 'blue']
class_colors = {'Target': 'steelblue', 'NotTarget': 'tomato'}

for ax, t in zip(axes, TYPES):
    ax.set_title(t)
    for c in CLASSES:
        folder = DATASET_ROOT / t / c
        images = list(folder.glob('*.jpg')) + list(folder.glob('*.JPG')) + list(folder.glob('*.png'))
        histograms = np.zeros(256)
        count = 0
        for img_path in images[:30]:  # échantillon 30 images pour la vitesse
            try:
                img = np.array(Image.open(img_path).convert('L').resize((224, 224)))
                hist, _ = np.histogram(img.flatten(), bins=256, range=(0, 256))
                histograms += hist
                count += 1
            except:
                pass
        if count > 0:
            histograms /= count
            ax.plot(histograms, color=class_colors[c], label=c, alpha=0.8)
    ax.legend()
    ax.set_xlabel('Intensité pixel')
    ax.set_ylabel('Fréquence moyenne')

fig.suptitle('Distribution luminosité moyenne — Target vs NotTarget', fontsize=13)
plt.tight_layout()
plt.show()